# Phase 2 — Data Representation (Prototyping Surface)

This notebook is where we **prototype the SQL** for the readmission dataset and
analyze each layer before freezing it into Dataform. The rule: we iterate on SQL
*against BigQuery* (not in pandas), pull the **result** back for EDA, and the
final query text drops verbatim into the matching `.sqlx` — table refs wrapped
in `ref()`.

We build in **DAG order**, mirroring the Dataform build so what we see here
matches what ships:

```
sources -> cohort -> cohort_split -> features -> features_clean -> analytics_dataset
```

The split happens **before** feature engineering and missingness handling — any
train-derived statistic (e.g. an imputation median) must come from training rows
only. That ordering is the core leakage guard, so we honor it here too: never
profile distributions or compute fill values on the full cohort.

| Surface | Owns |
|---|---|
| This notebook | EDA, SQL prototyping, distribution/null profiling, deriving assertion thresholds |
| Dataform `.sqlx` | The frozen logic, `ref()` wiring, the DAG, the data contract |
| `docs/workflow.md` | Back-filling placeholders (transformation strategy, feature list) |

## 0. Setup — BigQuery connection & config

Source dataset names are the **verified** PhysioNet values
(`mimiciv_3_1_*`), matching `dataform.json`. The **billing project** is *not*
hardcoded — it resolves from `PROJECT_ID` (env var or the repo-root `.env`, the
same file the shell scripts read), so no personal project ID is committed.
See the Dataform README's *"Running this yourself"* section for first-time setup.

Queries read from the public `physionet-data` project but are **billed** to
`BILLING_PROJECT`.


In [8]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from google.cloud import bigquery


def _resolve_billing_project() -> str:
    """Resolve the GCP billing project — no personal ID committed to git.

    Order: the PROJECT_ID env var, then the repo-root .env (the same file the
    shell scripts read — see .env.example). Kept dependency-free so the
    notebook runs without python-dotenv. See the Dataform README's
    "Running this yourself" section.
    """
    if os.environ.get("PROJECT_ID"):
        return os.environ["PROJECT_ID"]

    for directory in [Path.cwd(), *Path.cwd().resolve().parents]:
        env_file = directory / ".env"
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                if line.strip().startswith("PROJECT_ID="):
                    return line.split("=", 1)[1].strip()
            break

    raise RuntimeError(
        "PROJECT_ID is not set. Export it, or copy .env.example -> .env at the "
        "repo root and set PROJECT_ID to your GCP project."
    )


# --- Billing / connection ------------------------------------------------
# MIMIC-IV lives in the public `physionet-data` project; we read from there
# but queries are billed to this project.
BILLING_PROJECT = _resolve_billing_project()

# --- MIMIC-IV source datasets (PhysioNet public project) -----------------
MIMIC_PROJECT = "physionet-data"
MIMIC_HOSP = f"{MIMIC_PROJECT}.mimiciv_3_1_hosp"
MIMIC_ICU = f"{MIMIC_PROJECT}.mimiciv_3_1_icu"
MIMIC_ED = f"{MIMIC_PROJECT}.mimiciv_ed"
MIMIC_NOTE = f"{MIMIC_PROJECT}.mimiciv_note"

client = bigquery.Client(project=BILLING_PROJECT)


def run_sql(sql: str) -> pd.DataFrame:
    """Run BigQuery SQL and return the result as a DataFrame.

    Prototype layer SQL here; once it's right, the final text drops into the
    matching Dataform .sqlx with table refs wrapped in ref().
    """
    return client.query(sql).result().to_dataframe()


print(f"BigQuery client ready - billing project: {client.project}")


BigQuery client ready - billing project: trim-icon-498815-a0


**Tier B — needs a derivation:**

| Rule | Why |
|---|---|
| Administrative transfers / contiguous stays | Compare each admission to the patient's prior one; a re-admit within `CONTIGUOUS_HOURS` of the previous discharge is the same episode. **Decision: keep the first `hadm_id`, drop the continuation.** |
| Planned follow-up visits | **Decision: exclude `admission_type IN ('ELECTIVE', 'SURGICAL SAME DAY ADMISSION')`** — the only two scheduled-in-advance types. |

**Observation stays:** kept. The four `*OBSERVATION*` types are treated as real
inpatient admissions; the **LOS ≥ 1 day** gate is what removes the trivial ones,
not a blanket type exclusion.

The cell below profiles the categorical vocabularies (already run); the two cells
after build the spine in two steps — a **funnel** (count dropped at each gate, for
visibility) then the **skinny spine** itself. The final spine SQL backs
`staging/cohort.sqlx`.

In [4]:
# Categorical vocabulary — counts per distinct value for the three columns
# that drive Tier B (planned-visit + transfer logic). Read-only.
vocab_sql = f"""
SELECT 'admission_type'     AS column_name, admission_type     AS value, COUNT(*) AS n
FROM `{MIMIC_HOSP}.admissions` GROUP BY value
UNION ALL
SELECT 'admission_location' AS column_name, admission_location AS value, COUNT(*) AS n
FROM `{MIMIC_HOSP}.admissions` GROUP BY value
UNION ALL
SELECT 'discharge_location' AS column_name, discharge_location AS value, COUNT(*) AS n
FROM `{MIMIC_HOSP}.admissions` GROUP BY value
ORDER BY column_name, n DESC
"""

vocab = run_sql(vocab_sql)
for col in ["admission_type", "admission_location", "discharge_location"]:
    print(f"\n=== {col} ===")
    print(vocab[vocab.column_name == col][["value", "n"]].to_string(index=False))


=== admission_type ===
                      value      n
                   EW EMER. 177459
             EU OBSERVATION 119456
          OBSERVATION ADMIT  84437
                     URGENT  54929
SURGICAL SAME DAY ADMISSION  42898
         DIRECT OBSERVATION  24551
               DIRECT EMER.  21973
                   ELECTIVE  13130
     AMBULATORY OBSERVATION   7195

=== admission_location ===
                                 value      n
                        EMERGENCY ROOM 244179
                    PHYSICIAN REFERRAL 163228
                TRANSFER FROM HOSPITAL  56227
                 WALK-IN/SELF REFERRAL  42365
                       CLINIC REFERRAL  12965
                        PROCEDURE SITE   8518
TRANSFER FROM SKILLED NURSING FACILITY   6317
    INTERNAL TRANSFER TO OR FROM PSYCH   5837
                                  PACU   5734
             INFORMATION NOT AVAILABLE    402
           AMBULATORY SURGERY TRANSFER    255
                                  None      1


In [5]:
# --- Spine parameters (tunable, kept visible) ----------------------------
PLANNED_TYPES = ["ELECTIVE", "SURGICAL SAME DAY ADMISSION"]
CONTIGUOUS_HOURS = 24  # re-admit within 1 day of prior discharge = same episode

_planned_list = ", ".join(f"'{t}'" for t in PLANNED_TYPES)

# Shared base CTE — reused by both the funnel (below) and the spine (next cell)
# so the two can never drift. `prev_dischtime` uses LAG over the patient's full
# admission timeline to detect contiguous stays.
BASE_CTE = f"""
WITH base AS (
  SELECT
    a.subject_id, a.hadm_id, a.admittime, a.dischtime,
    p.anchor_age, a.hospital_expire_flag, a.deathtime, a.admission_type,
    TIMESTAMP_DIFF(a.dischtime, a.admittime, HOUR) AS los_hours,
    LAG(a.dischtime) OVER (PARTITION BY a.subject_id ORDER BY a.admittime) AS prev_dischtime
  FROM `{MIMIC_HOSP}.admissions` AS a
  JOIN `{MIMIC_HOSP}.patients`   AS p USING (subject_id)
),
flagged AS (
  SELECT *,
    (prev_dischtime IS NOT NULL
       AND TIMESTAMP_DIFF(admittime, prev_dischtime, HOUR) <= {CONTIGUOUS_HOURS}
    ) AS is_continuation
  FROM base
)
"""

# Funnel: cumulative survivors after each gate is applied in order.
funnel_sql = BASE_CTE + f"""
SELECT
  COUNT(*) AS s0_all_admissions,
  COUNTIF(anchor_age >= 18) AS s1_adult,
  COUNTIF(anchor_age >= 18 AND hospital_expire_flag = 0 AND deathtime IS NULL) AS s2_alive,
  COUNTIF(anchor_age >= 18 AND hospital_expire_flag = 0 AND deathtime IS NULL
          AND los_hours >= 24) AS s3_los_ge_1d,
  COUNTIF(anchor_age >= 18 AND hospital_expire_flag = 0 AND deathtime IS NULL
          AND los_hours >= 24 AND admission_type NOT IN ({_planned_list})) AS s4_unplanned,
  COUNTIF(anchor_age >= 18 AND hospital_expire_flag = 0 AND deathtime IS NULL
          AND los_hours >= 24 AND admission_type NOT IN ({_planned_list})
          AND NOT is_continuation) AS s5_spine
FROM flagged
"""

funnel = run_sql(funnel_sql).T.rename(columns={0: "rows"})
funnel["dropped"] = funnel["rows"].diff().fillna(0).astype(int) * -1
print(funnel.to_string())

                     rows  dropped
s0_all_admissions  546028        0
s1_adult           546028        0
s2_alive           534227    11801
s3_los_ge_1d       418550   115677
s4_unplanned       364738    53812
s5_spine           352699    12039


In [6]:
# The skinny spine: reuse BASE_CTE (same gates as the funnel, so no drift).
# Identity + timing only — demographics/label join on in later layers.
spine_sql = BASE_CTE + f"""
SELECT
  subject_id,
  hadm_id,
  admittime,
  dischtime
FROM flagged
WHERE anchor_age >= 18                                   -- adult
  AND hospital_expire_flag = 0 AND deathtime IS NULL     -- discharged alive
  AND los_hours >= 24                                    -- LOS >= 1 day
  AND admission_type NOT IN ({_planned_list})            -- exclude planned
  AND NOT is_continuation                                -- collapse contiguous stays
"""

spine = run_sql(spine_sql)

# Integrity checks — must match the funnel's s5_spine and be unique on hadm_id.
print(f"spine rows         : {len(spine):,}  (expected 352,699)")
print(f"distinct hadm_id   : {spine.hadm_id.nunique():,}  (must equal rows)")
print(f"distinct subject_id: {spine.subject_id.nunique():,}  (patients)")
print(f"null check         : {spine.isnull().sum().sum()} nulls")
spine.head()

spine rows         : 352,699  (expected 352,699)
distinct hadm_id   : 352,699  (must equal rows)
distinct subject_id: 164,847  (patients)
null check         : 0 nulls


,subject_id,hadm_id,admittime,dischtime
0,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00
1,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00
3,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00
4,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00


## 2. Deterministic split

Assign each admission to one of five disjoint groups via a **weighted** hash on
`subject_id`: `MOD(ABS(FARM_FINGERPRINT(subject_id)), 100)` → a 0–99 bucket,
mapped to groups by range.

**Weights — 70 / 14 / 14 / 1 / 1:**

| Group | Buckets (0–99) | Share |
|---|---|---|
| validation | 0–13 | 14% |
| test | 14–27 | 14% |
| prod_test | 28 | 1% |
| demo | 29 | 1% |
| **train** | 30–99 | **70%** |

- **All five groups are listed explicitly** in `SPLIT_RANGES` (train included),
  and a coverage guard asserts the ranges cover every bucket 0–99 — so the shares
  provably sum to 100% with nothing unassigned. (Requested 68/14/14/1/1 sums to
  98%; the leftover 2% goes to train → 70%.)
- **Keyed on `subject_id`, not `hadm_id`** — all of a patient's admissions land
  in one group. Leakage guard from `workflow.md` §2, enforced by
  `assertions/split_is_disjoint.sqlx`.
- **Deterministic + reproducible** — `FARM_FINGERPRINT` is a stable hash; same
  assignment every run, no stored seed.

This backs `staging/cohort_split.sqlx`.

In [7]:
# --- Split parameters (visible) ------------------------------------------
# Weighted split: hash subject_id into 0..SPLIT_RESOLUTION-1, map ranges to
# groups. Ranges are listed in bucket order and must cover every bucket.
SPLIT_RESOLUTION = 100  # bucket granularity (100 -> 1% steps)

# (group, upper_exclusive_bound) — each group claims [prev_bound, bound).
# train is listed explicitly and closes out the range at SPLIT_RESOLUTION.
SPLIT_RANGES = [
    ("validation", 14),   # buckets  0..13  -> 14%
    ("test",       28),   # buckets 14..27  -> 14%
    ("prod_test",  29),   # bucket  28      ->  1%
    ("demo",       30),   # bucket  29      ->  1%
    ("train",      100),  # buckets 30..99  -> 70%
]

# Coverage guard: bounds must be strictly increasing and the last must equal
# SPLIT_RESOLUTION, so every bucket (including train's) is accounted for.
_bounds = [b for _, b in SPLIT_RANGES]
assert _bounds == sorted(set(_bounds)), "bounds must be strictly increasing"
assert _bounds[-1] == SPLIT_RESOLUTION, "last bound must close the full range"

# Build the CASE ladder from the ranges (so SQL + table above can't drift).
_when = "\n".join(
    f"      WHEN h < {bound} THEN '{name}'" for name, bound in SPLIT_RANGES
)
_split_case = f"    CASE\n{_when}\n    END AS split_name"

# Reuse BASE_CTE + the validated spine gates, then hash-bucket on subject_id.
split_sql = BASE_CTE + f""",
spine AS (
  SELECT subject_id, hadm_id, admittime, dischtime
  FROM flagged
  WHERE anchor_age >= 18
    AND hospital_expire_flag = 0 AND deathtime IS NULL
    AND los_hours >= 24
    AND admission_type NOT IN ({_planned_list})
    AND NOT is_continuation
),
hashed AS (
  SELECT
    s.*,
    MOD(ABS(FARM_FINGERPRINT(CAST(s.subject_id AS STRING))), {SPLIT_RESOLUTION}) AS h
  FROM spine AS s
)
SELECT
  subject_id, hadm_id, admittime, dischtime,
  h AS split_bucket,
{_split_case}
FROM hashed
"""

split = run_sql(split_sql)

# Proportions by group — admissions and distinct patients.
summary = (
    split.groupby("split_name")
    .agg(admissions=("hadm_id", "size"), patients=("subject_id", "nunique"))
    .assign(pct=lambda d: (d.admissions / d.admissions.sum() * 100).round(1))
    .sort_values("admissions", ascending=False)
)
print(summary.to_string())

# Leakage guard (mirrors assertions/split_is_disjoint.sqlx): every subject_id
# must map to exactly one group.
spanning = split.groupby("subject_id").split_name.nunique()
print(f"\nrows: {len(split):,}   subjects in >1 group: {(spanning > 1).sum()}  (must be 0)")
print(f"unassigned (NULL) split_name: {split.split_name.isnull().sum()}  (must be 0)")

            admissions  patients   pct
split_name                            
train           247687    115468  70.2
test             49103     23126  13.9
validation       48924     23002  13.9
prod_test         3583      1646   1.0
demo              3402      1605   1.0

rows: 352,699   subjects in >1 group: 0  (must be 0)
unassigned (NULL) split_name: 0  (must be 0)
